In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath("..")  # because notebook is inside /notebooks
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)
print("Current working dir:", os.getcwd())


Project root added: f:\Movies and imp files from desktop\Programming\Python\ML- github\anomaly-based-ids
Current working dir: f:\Movies and imp files from desktop\Programming\Python\ML- github\anomaly-based-ids\notebooks


In [2]:
import pandas as pd
import numpy as np

from src.preprocess import preprocess_data
from src.train_isolation_forest import train_isolation_forest, save_model
from src.evaluate import get_anomaly_scores, choose_threshold, evaluate_model

print("All imports OK ✅")


All imports OK ✅


In [ ]:
from scipy.io import arff

train_path = os.path.join(PROJECT_ROOT, "data", "raw", "KDDTrain+.arff")
test_path  = os.path.join(PROJECT_ROOT, "data", "raw", "KDDTest+.arff")

data_train, meta_train = arff.loadarff("F:\\\Movies and imp files from desktop\\Programming\\Python\\ML- github\\anomaly-based-ids\\data\\raw\\KDDTrain+.txt")
data_test, meta_test = arff.loadarff("F:\\Movies and imp files from desktop\\Programming\\Python\\ML- github\\anomaly-based-ids\\data\\raw\\KDDTest+.txt")

df_train = pd.DataFrame(data_train)
df_test = pd.DataFrame(data_test)

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)


<>:6: SyntaxWarning: invalid escape sequence '\M'
<>:7: SyntaxWarning: invalid escape sequence '\M'
<>:6: SyntaxWarning: invalid escape sequence '\M'
<>:7: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Advait\AppData\Local\Temp\ipykernel_33220\3897898342.py:6: SyntaxWarning: invalid escape sequence '\M'
  data_train, meta_train = arff.loadarff("F:\Movies and imp files from desktop\Programming\Python\ML- github\anomaly-based-ids\data\raw\KDDTrain+.txt")
C:\Users\Advait\AppData\Local\Temp\ipykernel_33220\3897898342.py:7: SyntaxWarning: invalid escape sequence '\M'
  data_test, meta_test = arff.loadarff("F:\Movies and imp files from desktop\Programming\Python\ML- github\anomaly-based-ids\data\raw\KDDTest+.txt")
C:\Users\Advait\AppData\Local\Temp\ipykernel_33220\3897898342.py:6: SyntaxWarning: invalid escape sequence '\M'
  data_train, meta_train = arff.loadarff("F:\Movies and imp files from desktop\Programming\Python\ML- github\anomaly-based-ids\data\raw\KDDTrain+.txt")
C:\Users\Ad

OSError: [Errno 22] Invalid argument: 'F:\\Movies and imp files from desktop\\Programming\\Python\\ML- github\x07nomaly-based-ids\\data\raw\\KDDTrain+.txt'

In [ ]:
def decode_bytes_df(df):
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)
    return df

df_train = decode_bytes_df(df_train)
df_test = decode_bytes_df(df_test)

print("Decoded byte columns ✅")


In [ ]:
df_train["is_attack"] = df_train["label"].apply(lambda x: 0 if x == "normal" else 1)
df_test["is_attack"] = df_test["label"].apply(lambda x: 0 if x == "normal" else 1)

print("Train is_attack:\n", df_train["is_attack"].value_counts())
print("Test is_attack:\n", df_test["is_attack"].value_counts())


In [ ]:
X_train_processed, X_test_processed, preprocessor = preprocess_data(df_train, df_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape :", X_test_processed.shape)


In [ ]:
X_train_normal = X_train_processed[df_train["is_attack"] == 0]
print("Normal samples used for training:", X_train_normal.shape)

model = train_isolation_forest(X_train_normal)
save_model(model)

print("Model trained + saved ✅")


In [ ]:
train_scores = get_anomaly_scores(model, X_train_processed)
threshold = choose_threshold(train_scores, percentile=95)

print("Threshold (95th percentile):", threshold)


In [ ]:
scores, preds, cm, report = evaluate_model(
    model,
    X_test_processed,
    df_test["is_attack"],
    threshold
)

print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)


In [ ]:
tn, fp, fn, tp = cm.ravel()

print("TP (attacks caught):", tp)
print("FN (missed attacks):", fn)
print("FP (false alerts):", fp)
print("TN (normal traffic):", tn)


In [ ]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files",
    "num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate",
    "srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]


In [ ]:
df_train = pd.read_csv("F:\\Movies and imp files from desktop\\Programming\\Python\\ML- github\\anomaly-based-ids\\data\\raw\\KDDTrain+.txt", names= columns)
df_test = pd.read_csv("F:\\Movies and imp files from desktop\\Programming\\Python\\ML- github\\anomaly-based-ids\\data\\raw\KDDTest+.txt", names= columns)

In [ ]:
print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)

df_train.head()


In [ ]:
df_train['label'].value_counts().head(10)


In [ ]:
df_train['is_attack'] = df_train['label'].apply(lambda x:0 if x=='normal' else 1)
df_test['is_attack'] = df_test['label'].apply(lambda x:0 if x=='normal' else 1)

df_train['is_attack'].value_counts()
df_test['is_attack'].value_counts()


In [ ]:
df_train = df_train.drop(columns='difficulty')
df_test = df_test.drop(columns='difficulty')

In [ ]:
X_train_processed, X_test_processed, preprocessor = preprocess_data(
    df_train,
    df_test
)

print(X_train_processed.shape)
print(X_test_processed.shape)


In [ ]:
from src.train_isolation_forest import train_isolation_forest, save_model

X_train_normal = X_train_processed[df_train["is_attack"] == 0]

model = train_isolation_forest(X_train_normal)
save_model(model)

print("Saved ✅")





In [ ]:
from src.train_isolation_forest import train_isolation_forest, save_model

X_train_normal = X_train_processed[df_train['is_attack'] == 0]

model = train_isolation_forest(X_train_normal)
save_model(model)

print("Model trained + saved ✅")


In [ ]:
from src.evaluate import get_anomaly_scores, choose_threshold

train_scores = get_anomaly_scores(model, X_train_processed)
threshold = choose_threshold(train_scores, percentile=95)

print("Threshold:", threshold)


In [ ]:
from src.evaluate import evaluate_model

scores, preds, cm, report = evaluate_model(
    model,
    X_test_processed,
    df_test['is_attack'],
    threshold
)

print("Confusion Matrix:\n", cm)
print("\nReport:\n", report)
